# Nemotron v8 — minimal-defensive training notebook

**Goal:** finish a training run end-to-end on Kaggle without crashing.
We strip every optional optimization that has caused us trouble:

| component | status | why |
|---|---|---|
| Unsloth | OFF | Read-only-fs install crashes |
| Mamba CUDA fast path | OFF | Wheel mismatches kill load |
| cut-cross-entropy | OFF | TRL `entropy_from_logits(None)` crash |
| MoE tied LoRA | OFF | `_create_and_replace` API is fragile |
| flash-attn | OFF | Nemotron-H doesn't support it (must be eager) |
| chat template tricks | OFF | use the tokenizer's default |

**Required Kaggle inputs:**
- `metric/nemotron-3-nano-30b-a3b-bf16` (model)
- Your category-splits dataset (path auto-detected)

**Internet must be ON** in the Kaggle kernel — we install 4 packages from PyPI.
If you must run offline, see the `OFFLINE_FALLBACK_DIR` variable in cell 1.

Expected runtime: ~9–12 hrs for 1 epoch on 8.5k samples at seq 4096, batch 1×4.


In [ ]:
# ── Cell 1: Install only what Kaggle is missing ──────────────────────
# Strategy: keep Kaggle's bundled torch/transformers/accelerate untouched.
# Install trl/peft/datasets/bitsandbytes into an isolated dir we own.
import subprocess, sys, os
from pathlib import Path

TARGET_DIR = "/kaggle/working/packages"
os.makedirs(TARGET_DIR, exist_ok=True)
if TARGET_DIR not in sys.path:
    sys.path.insert(0, TARGET_DIR)

# Optional: if you've uploaded an offline wheels dataset, point this at it.
# When set and the dir exists, we install from there instead of PyPI.
OFFLINE_FALLBACK_DIR = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"

PKGS = ["trl", "peft", "datasets", "bitsandbytes"]

def _pip(args, label):
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--target", TARGET_DIR, "--no-deps"] + args
    try:
        subprocess.check_call(cmd)
        print(f"[ok] {label}")
        return True
    except Exception as e:
        print(f"[warn] {label} failed: {e}")
        return False

if os.path.isdir(OFFLINE_FALLBACK_DIR):
    print(f"Using offline wheels: {OFFLINE_FALLBACK_DIR}")
    _pip(["--no-index", "--find-links", OFFLINE_FALLBACK_DIR] + PKGS,
         f"offline install: {PKGS}")
else:
    print("Installing from PyPI (kernel needs internet ON)")
    _pip(PKGS, f"online install: {PKGS}")

# Resolve namespace-package .pth files we may have dropped
for pth in Path(TARGET_DIR).glob("*.pth"):
    try:
        with pth.open() as fp:
            rel = fp.read().strip()
            p = pth.parent / rel
            if p.exists() and str(p) not in sys.path:
                sys.path.append(str(p))
    except Exception:
        pass

print("[ok] cell 1 done")


In [ ]:
# ── Cell 2: Imports + sanity ─────────────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, time, zipfile, glob
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f"PyTorch       : {torch.__version__}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
import transformers
print(f"transformers  : {transformers.__version__}")

# Nemotron-H ships its own modeling_nemotron_h.py via trust_remote_code.
# It tries to use a CUDA "fast path" (mamba_ssm/causal_conv1d) which often
# isn't installed cleanly. We DISABLE the fast path globally — slower but
# rock-solid. The Python-fallback rmsnorm is monkey-patched in cell 4.


In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
MAX_SEQ_LEN  = 4096      # safe on 95GB VRAM with bf16+GC, no fast path
NUM_EPOCHS   = 1
BATCH_SIZE   = 1
GRAD_ACCUM   = 4         # eff batch = 4
LR           = 2e-4
WARMUP_STEPS = 50

MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CATEGORY_FILES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl",
    "train_cot_numeral.jsonl",
    "train_cot_unit_conversion.jsonl",
]
DATA_DIR_CANDIDATES = [
    "/kaggle/input/nemotron-categorical-splits",
    "/kaggle/input/nemotron-categorical-splits/all_categorical_splits",
    "/kaggle/input/all-categorical-splits",
    "/kaggle/input/all-categorical-splits/all_categorical_splits",
    "/kaggle/input/nemotron-cot-categorical/all_categorical_splits",
    "/kaggle/input/nemotron-cot-categorical",
]

print(f"Config: epochs={NUM_EPOCHS}, batch={BATCH_SIZE}x{GRAD_ACCUM}, "
      f"seq={MAX_SEQ_LEN}, rank={LORA_RANK}, lr={LR}")


In [ ]:
# ── Cell 4: Load + format data ───────────────────────────────────────
data_dir = None
for c in DATA_DIR_CANDIDATES:
    if c and os.path.isdir(c) and any(os.path.exists(os.path.join(c, f)) for f in CATEGORY_FILES):
        data_dir = c
        break
assert data_dir, f"no data dir found; searched: {DATA_DIR_CANDIDATES}"
print(f"Data dir: {data_dir}")

records = []
for fname in CATEGORY_FILES:
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [skip] {fname}")
        continue
    n = 0
    with open(fpath) as f:
        for line in f:
            if not line.strip():
                continue
            records.append(json.loads(line))
            n += 1
    print(f"  {n:>5} from {fname}")
print(f"Total raw records: {len(records)}")

# Tokenizer first — we need it to format chat templates
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = []
fallbacks = 0
for rec in records:
    msgs = [m for m in rec["messages"] if m["role"] != "system"]
    try:
        text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=False)
    except Exception:
        fallbacks += 1
        text = (f"<|im_start|>user\n{msgs[0]['content']}<|im_end|>\n"
                f"<|im_start|>assistant\n{msgs[-1]['content']}<|im_end|>")
    texts.append(text)

if fallbacks:
    print(f"  [warn] used ChatML fallback for {fallbacks} records")

ds = Dataset.from_dict({"text": texts})

# Drop oversized samples (cheaper than truncation; we keep most of bit_manipulation
# anyway because we can use seq=4096; if you need more coverage raise MAX_SEQ_LEN)
def _len(ex):
    return {"_len": len(tokenizer(ex["text"], truncation=False)["input_ids"])}
ds = ds.map(_len, desc="counting tokens")
before = len(ds)
ds = ds.filter(lambda x: x["_len"] <= MAX_SEQ_LEN, desc="dropping oversized")
ds = ds.remove_columns(["_len"])
print(f"Kept {len(ds)} / {before} samples (dropped {before - len(ds)} > {MAX_SEQ_LEN} tokens)")
print(f"\nFirst sample (300 chars):\n{ds[0]['text'][:300]}")


In [ ]:
# ── Cell 5: Load base model + LoRA ───────────────────────────────────
print("Loading base model (bf16, eager attention) — this takes ~3 min...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
    attn_implementation="eager",   # Nemotron-H requires eager
)
model.gradient_checkpointing_enable()

# DISABLE the Mamba CUDA fast path globally — its wheel install is fragile.
# The Python rmsnorm fallback works fine, just uses more VRAM and is slower.
import sys
for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        mod.is_fast_path_available = False
        print(f"  disabled fast path in {name}")

# Patch rmsnorm_fn fallback (mamba_ssm Python path) where present.
import torch.nn.functional as F
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    var = x.pow(2).mean(-1, keepdim=True)
    y = x * torch.rsqrt(var + eps)
    out = y * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)

# Only patch mamba_ssm modules — touching transformers triggers torchvision import
for name, mod in list(sys.modules.items()):
    if any(k in name.lower() for k in ("mamba_ssm", "selective_scan")):
        if hasattr(mod, "rmsnorm_fn"):
            mod.rmsnorm_fn = _pure_rmsnorm_fn
            print(f"  patched rmsnorm_fn in {name}")

print("Model loaded.")

# Apply LoRA
lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",   # Attention + Mamba + MLP + MoE
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


In [ ]:
# ── Cell 6: Train ─────────────────────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",            # vanilla — no fused, no 8bit, Blackwell-stable
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,                  # eager attn doesn't support packing safely
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=args,
)

print(f"Steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else '?'}")
print("Starting training...")

t0 = time.time()
trainer.train()
print(f"Done. {(time.time() - t0)/3600:.2f} hrs   "
      f"peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")


In [ ]:
# ── Cell 7: Save adapter + zip submission ────────────────────────────
trainer.model.save_pretrained(OUTPUT_DIR)

# Set canonical base_model_name_or_path so vLLM eval can locate the base
config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

# Zip exactly the two files vLLM needs
zip_path = "/kaggle/working/submission.zip"
required = {"adapter_config.json", "adapter_model.safetensors"}
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        if fname in required:
            zf.write(os.path.join(OUTPUT_DIR, fname), arcname=fname)

with zipfile.ZipFile(zip_path) as zf:
    contents = sorted(zf.namelist())
print(f"Zip contents: {contents}")
print(f"Zip size    : {os.path.getsize(zip_path)/1024/1024:.1f} MB")
assert set(contents) == required, f"unexpected zip contents: {contents}"

# Quick health check on the saved weights
try:
    from safetensors import safe_open
    with safe_open(os.path.join(OUTPUT_DIR, "adapter_model.safetensors"),
                   framework="pt") as f:
        norms = [f.get_tensor(k).norm().item() for k in list(f.keys())[:5]]
    print(f"First 5 weight norms: {[f'{n:.4f}' for n in norms]}")
    if all(n < 0.001 for n in norms):
        print("WARNING: norms near zero — DO NOT submit.")
    else:
        print("Adapter looks healthy. Ready to submit.")
except Exception as e:
    print(f"weight check skipped: {e}")
